# Track 3: the three missing evals, one run

Evaluation only, on the **shipped** `sft4` adapter. Nothing is trained here.

| # | Run | What it measures | ~time |
|---|---|---|---|
| 1 | `sft4_shift_hard30` | Same 30 hard tasks, every tool and parameter renamed. Separates "learned the task" from "learned the format". | 35 min |
| 2 | `sft4_devanagari30` | Same six families written in Devanagari, Latin names and ISO codes. Untested script. | 30 min |
| 3 | `base_shift_hard30` | Control: untuned model on the shifted schema. If base also scores near 29/30 the shift is not a real test. | 35 min |

Order matters: the two headline runs go first, the control last, so a quota stop still leaves usable numbers.
Every run uploads itself to `primce/lexsi-runs` and is skipped if its `summary.json` already exists, so re-running this cell resumes instead of repeating.

Not in this notebook: the hardened prompt on hard-90. Already measured, 86/90, run folder `sft4_hard90_hardened`.


In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false

# ---- setup ----------------------------------------------------------------
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git || { echo "SETUP FAILED: clone"; exit 1; }
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py || { echo "SETUP FAILED: verify_all"; exit 1; }
test -f hinglish_agent/schema_shift.py || { echo "SETUP FAILED: schema_shift.py missing"; exit 1; }
test -f data/tasks_devanagari_eval.jsonl || { echo "SETUP FAILED: devanagari tasks missing"; exit 1; }

python -c "import os;from huggingface_hub import snapshot_download as d;d('primce/tiny-aya-hinglish-lora',allow_patterns=['sft4_adapter/*'],local_dir='runs/_adapter_dl',token=os.environ['HF_TOKEN'])"
test -f runs/_adapter_dl/sft4_adapter/adapter_config.json || { echo "SETUP FAILED: adapter missing"; exit 1; }
echo "SETUP OK"

# ---- runs -----------------------------------------------------------------
run_one () {
  name="$1"; shift
  if [ -f "runs/$name/summary.json" ]; then
    echo "SKIP $name (summary already exists)"
    return 0
  fi
  echo "=== RUN $name ==="
  if python scripts/run_baseline.py "$@" --no-4bit --out "runs/$name"; then
    python scripts/job_upload.py --run "$name" || echo "UPLOAD FAILED $name"
  else
    echo "EVAL FAILED $name"
  fi
}

run_one sft4_shift_hard30 \
  --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_hard_eval.jsonl \
  --schema-shift

run_one sft4_devanagari30 \
  --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_devanagari_eval.jsonl

run_one base_shift_hard30 \
  --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hard_eval.jsonl \
  --schema-shift

echo "ALL CELLS DONE"


In [ ]:
import json
from pathlib import Path

keys = ("n", "success", "tac", "arr", "answer_ok", "strict_rate", "calls",
        "parse_failures", "seconds", "schema_shift", "hardened_prompt")

for name in ("sft4_shift_hard30", "sft4_devanagari30", "base_shift_hard30"):
    path = Path("lexsi-agentic-hinglish/runs") / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(name, {k: summary[k] for k in keys if k in summary})
    for family, row in (summary.get("families") or {}).items():
        print(f"   {family}: n={row.get('n')} success={row.get('success')}")
